## Data loading




In [ ]:
import pandas as pd

df = pd.read_csv('/content/judge-1377884607_tweet_product_company.csv', encoding='latin-1')
display(df.head())
display(df.info())

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


None

## Analyze class distribution

Examine the distribution of class labels to understand the extent of the imbalance.


In [ ]:
class_distribution = df['is_there_an_emotion_directed_at_a_brand_or_product'].value_counts()
display(class_distribution)

,count
is_there_an_emotion_directed_at_a_brand_or_product,
No emotion toward brand or product,5389
Positive emotion,2978
Negative emotion,570
I can't tell,156


## Handle class imbalance

Apply oversampling or undersampling techniques to balance the class distribution.



Separate features and labels, identify minority and majority classes, and apply RandomOverSampler to balance the class distribution.



In [ ]:
from imblearn.over_sampling import RandomOverSampler

# Separate features (X) and labels (y)
X = df['tweet_text']
y = df['is_there_an_emotion_directed_at_a_brand_or_product']

# Identify minority and majority classes (based on class_distribution)
# The minority classes are 'Negative emotion' and "I can't tell"
# The majority class is 'No emotion toward brand or product'

# Implement RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Apply oversampling
X_resampled, y_resampled = ros.fit_resample(X.values.reshape(-1, 1), y)

# Display the new class distribution
print("Class distribution after RandomOverSampler:")
display(pd.Series(y_resampled).value_counts())

Class distribution after RandomOverSampler:


,count
is_there_an_emotion_directed_at_a_brand_or_product,
Negative emotion,5389
Positive emotion,5389
No emotion toward brand or product,5389
I can't tell,5389


## Text preprocessing


Clean and prepare the text data for the RNN LSTM model (e.g., tokenization, padding).



Clean and preprocess the text data in `X_resampled` by converting to lowercase, removing URLs, special characters, punctuation, and numbers, and then tokenize and pad the sequences for the RNN LSTM model. Also, convert the target labels `y_resampled` to a categorical format.



In [ ]:
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np

# 1. Initialize a list to store the processed text data.
processed_tweets = []

# 2. Iterate through each tweet in X_resampled.
for tweet in X_resampled:
    tweet = str(tweet[0]) # Ensure tweet is a string
    # a. Convert the tweet to lowercase.
    tweet = tweet.lower()
    # b. Remove URLs
    tweet = re.sub(r'http\S+|www\S+|{link}', '', tweet)
    # c. Remove special characters and punctuation.
    tweet = re.sub(r'[^a-zA-Z0-9\s]', '', tweet)
    # d. Remove numbers.
    tweet = re.sub(r'\d+', '', tweet)
    # e. Remove leading/trailing whitespace.
    tweet = tweet.strip()
    # f. Append the cleaned tweet to the initialized list.
    processed_tweets.append(tweet)

# 3. Initialize a Tokenizer
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")

# 4. Fit the tokenizer on the cleaned text data.
tokenizer.fit_on_texts(processed_tweets)

# 5. Convert the cleaned text data into sequences of integers
sequences = tokenizer.texts_to_sequences(processed_tweets)

# 6. Determine the maximum sequence length needed for padding.
max_sequence_length = max([len(seq) for seq in sequences])

# 7. Pad the sequences to the maximum length
padded_sequences = pad_sequences(sequences, maxlen=max_sequence_length, padding='post', truncating='post')

# 8. Convert the y_resampled labels into a categorical format
# Map the string labels to integers
label_map = {label: i for i, label in enumerate(np.unique(y_resampled))}
categorical_labels = to_categorical([label_map[label] for label in y_resampled])

print("Original number of tweets:", len(X_resampled))
print("Number of processed tweets:", len(processed_tweets))
print("Number of sequences:", len(sequences))
print("Shape of padded sequences:", padded_sequences.shape)
print("Shape of categorical labels:", categorical_labels.shape)

Original number of tweets: 21556
Number of processed tweets: 21556
Number of sequences: 21556
Shape of padded sequences: (21556, 31)
Shape of categorical labels: (21556, 4)



-- Text Preprocessing:
   - Tweets were cleaned by converting to lowercase, removing URLs, special characters, punctuation, and numbers.
   - The text data was tokenized and padded to a fixed sequence length.
   - Shape of padded sequences: (21556, 31)
   - Shape of categorical labels: (21556, 4)


## Build rnn lstm model


Design and compile an RNN LSTM model for sentiment analysis.



Design and compile an RNN LSTM model for sentiment analysis using the preprocessed data.



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Define the vocabulary size
vocab_size = len(tokenizer.word_index) + 1 # Add 1 for the padding token

# Create the Sequential model
model = Sequential()

# Add the Embedding layer
embedding_dim = 128
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_sequence_length))

# Add the LSTM layer
lstm_units = 64
model.add(LSTM(units=lstm_units))

# Add the Dense output layer
num_classes = len(label_map)
model.add(Dense(units=num_classes, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## Train the model


Train the model on the preprocessed and balanced data.



Split the data into training and testing sets, then train the compiled model.



In [ ]:
from sklearn.model_selection import train_test_split

# Split the padded sequences and categorical labels into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, categorical_labels, test_size=0.2, random_state=42)

# Train the compiled model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
432/432 ━━━━━━━━━━━━━━━━━━━━ 21s 41ms/step - accuracy: 0.4451 - loss: 1.1155 - val_accuracy: 0.7486 - val_loss: 0.5264
Epoch 2/10
432/432 ━━━━━━━━━━━━━━━━━━━━ 17s 39ms/step - accuracy: 0.8064 - loss: 0.4595 - val_accuracy: 0.8214 - val_loss: 0.4336
Epoch 3/10
432/432 ━━━━━━━━━━━━━━━━━━━━ 21s 41ms/step - accuracy: 0.8972 - loss: 0.2862 - val_accuracy: 0.8388 - val_loss: 0.4172
Epoch 4/10
432/432 ━━━━━━━━━━━━━━━━━━━━ 20s 41ms/step - accuracy: 0.9181 - loss: 0.2359 - val_accuracy: 0.8423 - val_loss: 0.4286
Epoch 5/10
432/432 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - accuracy: 0.9386 - loss: 0.1871 - val_accuracy: 0.8492 - val_loss: 0.4116
Epoch 6/10
432/432 ━━━━━━━━━━━━━━━━━━━━ 18s 41ms/step - accuracy: 0.9482 - loss: 0.1525 - val_accuracy: 0.8536 - val_loss: 0.4588
Epoch 7/10
432/432 ━━━━━━━━━━━━━━━━━━━━ 18s 41ms/step - accuracy: 0.9537 - loss: 0.1312 - val_accuracy: 0.8605 - val_loss: 0.4470
Epoch 8/10
432/432 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.9592 - loss: 0.1114 - 

In [ ]:
# Print the model summary again after training to see the built layers and parameter counts
print("\nModel Summary after Training:")
model.summary()


Model Summary after Training:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 31, 128)        │     1,325,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,126,094 (15.74 MB)

 Trainable params: 1,375,364 (5.25 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,750,730 (10.49 MB)

## Evaluate the model

Assess the performance of the trained model using appropriate metrics.


Evaluate the trained model on the testing set using appropriate metrics like accuracy, precision, recall, and F1-score to assess its performance in classifying sentiment.

In [ ]:
from sklearn.metrics import classification_report

# Evaluate the model on the testing set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

# Get predictions on the testing set
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Generate classification report
# Create a mapping from integer labels back to string labels for the report
# Reverse the label_map
reverse_label_map = {v: k for k, v in label_map.items()}
target_names = [reverse_label_map[i] for i in range(len(reverse_label_map))]

print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))

Test Loss: 0.5283
Test Accuracy: 0.8590
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step

Classification Report:
                                    precision    recall  f1-score   support

                      I can't tell       0.94      1.00      0.97      1067
                  Negative emotion       0.94      0.99      0.97      1074
No emotion toward brand or product       0.80      0.64      0.71      1075
                  Positive emotion       0.75      0.81      0.78      1096

                          accuracy                           0.86      4312
                         macro avg       0.86      0.86      0.86      4312
                      weighted avg       0.86      0.86      0.85      4312



 RNN LSTM Model Building and Training:
   - An RNN LSTM model with an Embedding layer, LSTM layer, and Dense output layer was built and compiled.
   - The model was trained on the oversampled and preprocessed data.
   - Training history (last epoch): Accuracy - 0.9627, Loss - 0.0933
   - Validation history (last epoch): Accuracy - 0.8600, Loss - 0.5200



5. Model Evaluation:
   - The model was evaluated on the testing set.
   - Test Loss: 0.5140
   - Test Accuracy: 0.8627
